# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arhxmz/Flyrank-Internship-ML/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [1]:
%pip -q install duckdb
import duckdb
import pandas as pd
import getpass

con = duckdb.connect()
hf_token = getpass.getpass("Paste your HF_TOKEN: ")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")


[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
REL = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"

In [3]:
work_df = con.sql(f"""
    SELECT
        f.content_hash_id,
        AVG(f.gsc_avg_position) AS avg_position,
        SUM(f.gsc_clicks) AS total_clicks,
        SUM(f.gsc_impressions) AS total_impressions
    FROM {REL} f
    WHERE f.gsc_data_available IS TRUE
    GROUP BY f.content_hash_id
""").df()
work_df.head()

,content_hash_id,avg_position,total_clicks,total_impressions
0,content_1855a661b4d36130,4.209227,1.0,429.0
1,content_61c495d39498082e,35.774832,1.0,1052.0
2,content_65f91a4d113c4849,64.359673,0.0,305.0
3,content_9df4886c4db1796b,9.144800,0.0,527.0
4,content_f55dce635f743a5e,10.125000,0.0,6.0


In [6]:
DIM_REL = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')"
dim = con.sql(f"""
    SELECT
        content_hash_id,
        DATE_DIFF('day', content_updated_date, DATE '2026-03-31') AS days_since_last_update
    FROM {DIM_REL}
""").df()

work_df = work_df.merge(dim, on="content_hash_id", how="inner")
work_df.head()

,content_hash_id,avg_position,total_clicks,total_impressions,days_since_last_update_x,days_since_last_update_y,days_since_last_update
0,content_1855a661b4d36130,4.209227,1.0,429.0,-48,-48,-48
1,content_61c495d39498082e,35.774832,1.0,1052.0,-83,-83,-83
2,content_65f91a4d113c4849,64.359673,0.0,305.0,-48,-48,-48
3,content_9df4886c4db1796b,9.144800,0.0,527.0,-48,-48,-48
4,content_f55dce635f743a5e,10.125000,0.0,6.0,-48,-48,-48


In [8]:
work_df["ctr"] = (work_df["total_clicks"] / work_df["total_impressions"]) * 100
work_df["ctr"].describe()

count    176738.000000
mean          0.459397
std           3.775992
min           0.000000
25%           0.000000
50%           0.000000
75%           0.215796
max         100.000000
Name: ctr, dtype: float64

In [9]:
work_df["staleness_bucket"] = pd.cut(
    work_df["days_since_last_update"],
    bins=[-1, 90, 180, 365, 10_000],
    labels=["<90", "90-180", "180-365", "365+"]
)
staleness_table = work_df.groupby("staleness_bucket", observed=True).agg(
    n=("content_hash_id", "count"),
    avg_clicks=("total_clicks", "mean")
)
print(staleness_table)

                      n  avg_clicks
staleness_bucket                   
<90               26370    2.915245
90-180             1325    0.980377
180-365             261    0.103448


In [10]:
work_df["position_bucket"] = pd.cut(
    work_df["avg_position"],
    bins=[0, 3, 10, 20, 50, 1000],
    labels=["1-3", "4-10", "11-20", "21-50", "50+"]
)
ctr_table = work_df.groupby("position_bucket", observed=True).agg(
    n=("content_hash_id", "count"),
    avg_ctr=("ctr", "mean")
)
print(ctr_table)

                     n   avg_ctr
position_bucket                 
1-3              16144  1.058938
4-10             81988  0.492605
11-20            32203  0.321119
21-50            33288  0.228715
50+              11681  0.090312


In [12]:
work_df["staleness_bucket"] = pd.cut(
    work_df["days_since_last_update"],
    bins=[-1, 90, 180, 365, 10_000],
    labels=["<90", "90-180", "180-365", "365+"]
)
staleness_table = work_df.groupby("staleness_bucket", observed=True).agg(
    n=("content_hash_id", "count"),
    avg_clicks=("total_clicks", "mean")
)
print(staleness_table)

                      n  avg_clicks
staleness_bucket                   
<90               26370    2.915245
90-180             1325    0.980377
180-365             261    0.103448


**Signal 1 — Staleness (behind the refresh flags): CONFIRMED.**
Bucketing by days since last update shows clicks fall sharply as pages age: <90 days averages 2.92 clicks, dropping to 0.98 at 90-180 days, and 0.10 at 180-365 days (n = 26,370 / 1,325 / 261). No pages in this slice are stale beyond 365 days. The direction matches what the refresh flag assumes — the older a page gets without an update, the less traffic it earns.

**Signal 2 — CTR vs. position (behind the CTR-fix logic): CONFIRMED.**
CTR drops steadily and sharply as position worsens: 1.06% at positions 1-3, down to 0.09% at position 50+ (n = 16,144 / 81,988 / 32,203 / 33,288 / 11,681). Roughly a 12x gap top to bottom — squarely confirms the CTR-fix logic's core assumption.

**The rule, in plain words:**
A page is worth reviewing for refresh if it's gone stale (no update in 90+ days) and still holds meaningful search visibility (real impressions/clicks) — refreshing a page nobody sees isn't worth an editor's time, and a page updated last week doesn't need attention yet either.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

**Score:** `stale_but_visible = is_stale × has_visibility × gsc_impressions` — a page only scores above zero if it's both stale (90+ days since last update) and still getting real search visibility.

**Reason code:** `"stale_but_visible"` — every scored row gets this one code, since it's the only rule this baseline encodes.

**Action label:** `"refresh"` for anything scoring above zero, `"no_action"` otherwise.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer i# ---------- Build the score ----------
work_df["is_stale"] = (work_df["days_since_last_update"] >= 90).astype(int)
work_df["has_visibility"] = (work_df["total_impressions"] > 0).astype(int)

work_df["score"] = work_df["is_stale"] * work_df["has_visibility"] * work_df["total_impressions"]

work_df["reason_code"] = "stale_but_visible"
work_df["action"] = work_df["score"].apply(lambda s: "refresh" if s > 0 else "no_action")

# ---------- Rank and write the queue ----------
ranked_queue = work_df.sort_values("score", ascending=False)[
    ["content_hash_id", "score", "reason_code", "action",
     "days_since_last_update", "total_impressions", "total_clicks", "avg_position"]
]

import os
os.makedirs("../../work/outputs", exist_ok=True)
ranked_queue.to_csv("../../work/outputs/baseline_action_score.csv", index=False)

print(f"Wrote {len(ranked_queue)} rows. {(ranked_queue['action']=='refresh').sum()} flagged for refresh.")
ranked_queue.head(10)n the cell ABOVE this one — typing sentences here breaks Run All.


SyntaxError: invalid character '—' (U+2014) (817707328.py, line 22)

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.